In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error

ratings = pd.read_csv('../data/ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)

n = len(ratings_sorted)
train_end = int(0.7 * n)
val_end = int(0.8 * n)
train_df = ratings_sorted.iloc[:train_end].copy()
val_df = ratings_sorted.iloc[train_end:val_end].copy()
test_df = ratings_sorted.iloc[val_end:].copy()
global_mean = train_df['rating'].mean()

genre_cols = ['unknown','Action','Adventure','Animation','Children','Comedy','Crime','Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery',
              'Romance','Sci-Fi','Thriller','War','Western']
item_cols = ['item_id','title','release_date','video_date','imdb_url'] + genre_cols

movies = pd.read_csv('../data/ml-100k/u.item', sep='|', names=item_cols, encoding='latin-1')
movies[genre_cols] = movies[genre_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

print(f'Фильмов: {len(movies)}')
print(movies[['item_id','title','release_date'] + genre_cols[:5]].head())

Фильмов: 1682
   item_id              title release_date  unknown  Action  Adventure  \
0        1   Toy Story (1995)  01-Jan-1995        0       0          0   
1        2   GoldenEye (1995)  01-Jan-1995        0       1          1   
2        3  Four Rooms (1995)  01-Jan-1995        0       0          0   
3        4  Get Shorty (1995)  01-Jan-1995        0       1          0   
4        5     Copycat (1995)  01-Jan-1995        0       0          0   

   Animation  Children  
0          1         1  
1          0         0  
2          0         0  
3          0         0  
4          0         0  


# Feature engineering

In [20]:
movies['year'] = pd.to_datetime(movies['release_date'], format='%d-%b-%Y', errors='coerce').dt.year
movies['year'] = movies['year'].fillna(movies['year'].median())

print(movies[['title', 'year']].head())
print(f'Диапозон годов: {movies['year'].min():.0f} - {movies['year'].max():.0f}')

               title    year
0   Toy Story (1995)  1995.0
1   GoldenEye (1995)  1995.0
2  Four Rooms (1995)  1995.0
3  Get Shorty (1995)  1995.0
4     Copycat (1995)  1995.0
Диапозон годов: 1922 - 1998


In [21]:
item_stats = train_df.groupby('item_id')['rating'].agg(['mean', 'count'])
item_stats.columns = ['item_mean_rating', 'item_popularity']

user_stats = train_df.groupby('user_id')['rating'].agg(['mean', 'count'])
user_stats.columns = ['user_mean_rating', 'user_activity']

item_stats['item_popularity_log'] = np.log1p(item_stats['item_popularity']) # моя идея логарифмирования из EDA
user_stats['user_activity_log'] = np.log1p(user_stats['user_activity'])

print(item_stats.head())
print(user_stats.head())

         item_mean_rating  item_popularity  item_popularity_log
item_id                                                        
1                3.906433              342             5.837730
2                3.301075               93             4.543295
3                3.027778               72             4.290459
4                3.506579              152             5.030438
5                3.323077               65             4.189655
         user_mean_rating  user_activity  user_activity_log
user_id                                                    
1                3.596838            253           5.537334
5                2.874286            175           5.170484
6                3.635071            211           5.356586
8                3.796610             59           4.094345
9                4.272727             22           3.135494


In [23]:
print(movies[genre_cols].dtypes.unique())

[dtype('int64')]


In [24]:
train_with_genres = train_df.merge(movies[['item_id']+ genre_cols], on='item_id', how='left')
genre_mean_rating = {}

for g in genre_cols:
    mask = train_with_genres[g] == 1
    genre_mean_rating[g] = train_with_genres.loc[mask, 'rating'].mean() if mask.sum() > 0 else global_mean
genre_mean_rating = pd.Series(genre_mean_rating)

def expected_rating_by_genres(row):
    active = [g for g in genre_cols if row[g] == 1]
    return np.mean([genre_mean_rating[g] for g in active]) if active else global_mean

movies['genre_expected_rating'] = movies.apply(expected_rating_by_genres, axis=1)

print(genre_mean_rating.sort_values(ascending=False).round(3))
print(movies[['title', 'genre_expected_rating']].head())

Film-Noir      3.907
War            3.825
Drama          3.691
Documentary    3.666
Crime          3.656
Mystery        3.647
Romance        3.637
Western        3.593
Sci-Fi         3.580
Animation      3.573
Thriller       3.520
Musical        3.514
Adventure      3.503
Action         3.493
Comedy         3.399
Children       3.329
Horror         3.269
Fantasy        3.215
unknown        3.200
dtype: float64
               title  genre_expected_rating
0   Toy Story (1995)               3.433443
1   GoldenEye (1995)               3.505383
2  Four Rooms (1995)               3.520093
3  Get Shorty (1995)               3.527566
4     Copycat (1995)               3.622515


In [25]:
year_median = movies['year'].median()

print(f"global_mean = {global_mean:.4f}")
print(f"year_median = {year_median:.0f}")

global_mean = 3.5300
year_median = 1995


In [31]:
def build_features(df):
    d = df.merge(movies[['item_id', 'year', 'genre_expected_rating']+ genre_cols], on='item_id', how='left')
    d = d.merge(item_stats, on='item_id', how='left')
    d = d.merge(user_stats, on='user_id', how='left')

    d['item_mean_rating'] = d['item_mean_rating'].fillna(d['genre_expected_rating']).fillna(global_mean)
    d['user_mean_rating'] = d['user_mean_rating'].fillna(global_mean)

    for col in ['item_popularity', 'user_activity', 'item_popularity_log', 'user_activity_log']:
        d[col] = d[col].fillna(0)

    d['year'] = d['year'].fillna(year_median)
    for g in genre_cols:
        d[g] = d[g].fillna(0)

    feature_cols = genre_cols + ['year', 'item_mean_rating', 'user_mean_rating',
                                 'item_popularity_log', 'user_activity_log']
    X = d[feature_cols].astype(float)
    y = d['rating'].astype(float)
    return X, y

X_train, y_train = build_features(train_df)
X_val, y_val = build_features(val_df)
X_test, y_test = build_features(test_df)


print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)
print(X_train.head())

assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
print("Пропусков нет")

X_train: (70000, 24) | X_val: (10000, 24) | X_test: (20000, 24)
   unknown  Action  Adventure  Animation  Children  Comedy  Crime  \
0      0.0     0.0        0.0        0.0       0.0     1.0    0.0   
1      0.0     0.0        0.0        0.0       0.0     0.0    0.0   
2      0.0     1.0        0.0        0.0       0.0     0.0    0.0   
3      0.0     0.0        0.0        0.0       0.0     0.0    0.0   
4      0.0     1.0        1.0        0.0       0.0     1.0    0.0   

   Documentary  Drama  Fantasy  ...  Romance  Sci-Fi  Thriller  War  Western  \
0          0.0    0.0      0.0  ...      1.0     0.0       0.0  0.0      0.0   
1          0.0    1.0      0.0  ...      1.0     0.0       0.0  1.0      0.0   
2          0.0    0.0      0.0  ...      0.0     1.0       1.0  0.0      0.0   
3          0.0    0.0      0.0  ...      1.0     0.0       1.0  0.0      0.0   
4          0.0    0.0      0.0  ...      1.0     0.0       0.0  0.0      0.0   

     year  item_mean_rating  user_mean_r

In [32]:
d_debug = train_df.merge(movies[['item_id', 'year', 'genre_expected_rating'] + genre_cols], on='item_id', how='left')
d_debug = d_debug.merge(item_stats, on='item_id', how='left')
d_debug = d_debug.merge(user_stats, on='user_id', how='left')
print(d_debug.columns.tolist())
print(d_debug.shape)
d_debug.head()

['user_id', 'item_id', 'rating', 'timestamp', 'year', 'genre_expected_rating', 'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'item_mean_rating', 'item_popularity', 'item_popularity_log', 'user_mean_rating', 'user_activity', 'user_activity_log']
(70000, 31)


,user_id,item_id,rating,timestamp,year,genre_expected_rating,unknown,Action,Adventure,Animation,...,Sci-Fi,Thriller,War,Western,item_mean_rating,item_popularity,item_popularity_log,user_mean_rating,user_activity,user_activity_log
0,259,255,4,874724710,1997.0,3.517639,0,0,0,0,...,0,0,0,0,3.384615,117,4.770685,3.897436,39,3.688879
1,259,286,4,874724727,1996.0,3.717710,0,0,0,0,...,0,0,1,0,3.740061,327,5.793014,3.897436,39,3.688879
2,259,298,4,874724754,1997.0,3.531206,0,1,0,0,...,1,1,0,0,3.838028,142,4.962845,3.897436,39,3.688879
3,259,185,4,874724781,1960.0,3.475268,0,0,0,0,...,0,1,0,0,4.069767,172,5.153292,3.897436,39,3.688879
4,259,173,4,874724843,1987.0,3.507833,0,1,1,0,...,0,0,0,0,4.219008,242,5.493061,3.897436,39,3.688879
